In [3]:
import os
import cv2
import numpy as np
from typing import Optional
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    BatchNormalization,
    MaxPooling2D,
    Dropout,
    Flatten,
    Dense
)

In [19]:
print("      HANDWRITTEN CHARACTER RECOGNIZER — USER GUIDE      ")
print("=" * 55)
print("  [Left-Click + Drag]  : Draw character on canvas")
print("  [Right-Click + Drag] : Erase specific strokes")
print("  [Release Mouse]      : Auto-run model prediction")
print("  [Press 'C']          : Clear the canvas")
print("  [Press 'Esc']        : Exit application")

      HANDWRITTEN CHARACTER RECOGNIZER — USER GUIDE      
  [Left-Click + Drag]  : Draw character on canvas
  [Right-Click + Drag] : Erase specific strokes
  [Release Mouse]      : Auto-run model prediction
  [Press 'C']          : Clear the canvas
  [Press 'Esc']        : Exit application


In [20]:
# =========================================================
# CONFIGURATION & MODEL ARCHITECTURE
# =========================================================
LABELS = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
    'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9'
]

def build_model():
    return Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, (3, 3), activation="relu", padding="same"), BatchNormalization(),
        Conv2D(32, (3, 3), activation="relu", padding="same"), BatchNormalization(),
        MaxPooling2D((2, 2)), Dropout(0.25),
        
        Conv2D(64, (3, 3), activation="relu", padding="same"), BatchNormalization(),
        Conv2D(64, (3, 3), activation="relu", padding="same"), BatchNormalization(),
        MaxPooling2D((2, 2)), Dropout(0.25),
        
        Conv2D(128, (3, 3), activation="relu", padding="same"), BatchNormalization(),
        MaxPooling2D((2, 2)), Dropout(0.3),
        
        Flatten(),
        Dense(512, activation="relu"), BatchNormalization(), Dropout(0.4),
        Dense(256, activation="relu"), BatchNormalization(), Dropout(0.3),
        Dense(128, activation="relu"), BatchNormalization(), Dropout(0.2),
        Dense(36, activation="softmax")
    ])

# Initialize Canvas (500x500 Whiteboard)
canvas = np.ones((500, 500, 3), dtype=np.uint8) * 255
drawing = False
last_pt = None
top_preds = []

# Load Model
model = build_model()
weights_path = "models/best_val_loss_model.weights.h5"
if os.path.exists(weights_path):
    model.load_weights(weights_path)

# =========================================================
# HELPER & PREDICTION FUNCTIONS
# =========================================================
def predict_canvas():
    """Extracts drawing, resizes to 28x28, and runs inference."""
    global top_preds
    gray = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
    inverted = cv2.bitwise_not(gray)

    # Find drawing boundaries
    coords = cv2.findNonZero(inverted)
    if coords is None:
        top_preds = []
        return

    x, y, w, h = cv2.boundingRect(coords)
    crop = inverted[y:y+h, x:x+w]

    # Square Padding
    max_dim = max(w, h)
    padded = cv2.copyMakeBorder(crop, (max_dim-h)//2, (max_dim-h)//2, 
                                (max_dim-w)//2, (max_dim-w)//2, cv2.BORDER_CONSTANT, value=0)

    # Normalize to 28x28
    resized = cv2.resize(padded, (28, 28), interpolation=cv2.INTER_AREA)
    norm = resized.astype(np.float32) / 255.0
    input_tensor = np.expand_dims(norm, axis=(0, -1))

    preds = model.predict(input_tensor, verbose=0)[0]
    top_indices = np.argsort(preds)[-3:][::-1]
    top_preds = [(LABELS[i], preds[i] * 100) for i in top_indices if preds[i] > 0.01]

def draw(event, x, y, flags, param):
    """Mouse event listener for brush and eraser."""
    global drawing, last_pt, canvas
    if event in (cv2.EVENT_LBUTTONDOWN, cv2.EVENT_RBUTTONDOWN):
        drawing = True
        last_pt = (x, y)
    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        color = (0, 0, 0) if flags & cv2.EVENT_FLAG_LBUTTON else (255, 255, 255)
        radius = 10 if color == (0, 0, 0) else 20
        cv2.line(canvas, last_pt, (x, y), color, radius * 2)
        cv2.circle(canvas, (x, y), radius, color, -1)
        last_pt = (x, y)
    elif event in (cv2.EVENT_LBUTTONUP, cv2.EVENT_RBUTTONUP):
        drawing = False
        last_pt = None
        predict_canvas()  # Auto-predict when mouse released

# =========================================================
# MAIN APP LOOP
# =========================================================
cv2.namedWindow("Character Recognizer")
cv2.setMouseCallback("Character Recognizer", draw)

while True:
    # 1. Create Layout Window (Canvas + Sidebar)
    ui_panel = np.full((500, 250, 3), (35, 35, 35), dtype=np.uint8)

    # Header
    cv2.putText(ui_panel, "PREDICTIONS", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.line(ui_panel, (20, 50), (230, 50), (60, 60, 60), 1)

    # Display Top 3 Predictions
    colors = [(0, 220, 255), (180, 180, 180), (120, 120, 120)]
    for i, item in enumerate(top_preds):
        lbl, conf = item
        y_pos = 100 + (i * 60)
        cv2.rectangle(ui_panel, (20, y_pos - 30), (230, y_pos + 15), (25, 25, 25), -1)
        cv2.putText(ui_panel, f"#{i+1} {lbl}", (30, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.8, colors[i], 2)
        cv2.putText(ui_panel, f"{conf:.1f}%", (150, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1)

    # Instructions Legend
    cv2.putText(ui_panel, "Left Click : Draw", (20, 420), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1)
    cv2.putText(ui_panel, "Right Click: Erase", (20, 440), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1)
    cv2.putText(ui_panel, "Press [C]  : Clear Board", (20, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (235, 155, 52), 1)

    # Combine Canvas & Sidebar
    display = np.hstack((canvas, ui_panel))
    cv2.imshow("Character Recognizer", display)

    # Keybinds
    key = cv2.waitKey(1) & 0xFF
    if key == 27:  # ESC
        break
    elif key in (ord('c'), ord('C')):
        canvas[:] = 255
        top_preds = []

cv2.destroyAllWindows()